# Methodology Testing — every A/B and validation test in one notebook
Maize 2024 pipeline, Kenya and Ethiopia. Each section is one test: what it asks, the code that runs it, and the decision rule.

**Where to start:** put the `planting_pipeline` folder on your Google Drive, run Setup, then run any test section on its own. GEE-stage cells start Earth Engine exports to `Drive/planting_outputs`; scoring cells read those CSVs back, so wait for the export task to finish (Tasks tab in the EE Code Editor, or the cell below that polls).

| # | Test | Question | Verdict on record |
|---|---|---|---|
| T1 | Fusion (onset) | cue fusion vs ubESTARFM at 250 m | cue kept; ubESTARFM RETIRED (−3 to −4 pts) |
| T2 | Resolution (onset) | 10 m vs 250 m | −11.2 pts at 250 m; resolution is the dominant lever |
| T3 | LTN prior ablation | does the prior sharpen SOS? | +4.5 pts long rains; short rains confirmation-limited |
| T4 | Growing period | fixed 120 d vs zone-aware LGP | fixed 120 d kept |
| T5 | Ym recalibration | fit Ym to HarvestStat, 70/30 | KE-LR 2.56 · KE-SR 2.1 · ET 4.6 |
| T6 | Per-zone Ym | single Ym vs highland 3.7 / other 2.3 | per-zone adopted (MAE 0.568 → 0.472) |
| T7 | Soil bucket (WHC) | uniform 100 mm vs SoilGrids/Saxton | tie long rains; significant win short rains |
| T8 | Season length | fixed 120 d vs GDD thermal clock | highlands run ~170+ d; clock feeds stage windows |
| T9 | Biomass route (DMP) | MODIS-GPP yield vs CPI yield | good ranking, hot level; covariate not product |
| T10 | Planting vs farmers | dekad skill vs ~4.8 M records | county MAE 1.02 dk, 92.9% within ±2 |


## Setup

### Stage 0 · What this notebook is

**Every A/B and validation test in one place**, for the 2024 maize pipeline in Kenya and Ethiopia. Each
section is one test: the question, the code that reproduces it, and the verdict on record.

**Most re-run switches are off by default.** The heavy stages are guarded by flags such as
`RERUN_UBESTARFM` and `RUN_WHC_EXPORTS`, set to `False`, so the notebook runs end to end in seconds and
prints the recorded findings. Set a flag to `True` only when you intend to reproduce that test, and
expect Earth Engine exports that run for hours.

The same tests, one notebook each and with more detail, are in `Methodology_Testing/01` to `05`.

In [4]:
!pip -q install earthengine-api geemap pandas scipy 2>/dev/null
print('installed.')

installed.


### Stage 0b · Earth Engine

`EE ready: ok`. Only needed for the re-run flags; the scoring cells read CSVs. Use
`indigo-proxy-484220-q8` if you are going to export, since the queue is per project.

In [5]:
import ee
PROJECT="ee-manzikye"
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate(); ee.Initialize(project=PROJECT)
print("EE ready:", ee.String("ok").getInfo())

EE ready: ok


### Stage 0c · Drive, and where the results live

Two folders matter. `Cropyield-Data/` inside the pipeline holds the scoring inputs and outputs, and
`DRIVE_OUT` is where Earth Engine exports land. The helper below searches both.

In [7]:
from google.colab import drive; drive.mount("/content/drive")
import sys, os
PIPE_DIR="/content/drive/MyDrive/planting_pipeline"   # adjust if needed
assert os.path.isdir(PIPE_DIR), f"Upload planting_pipeline to Drive; not at {PIPE_DIR}"
sys.path.insert(0, PIPE_DIR); os.chdir(PIPE_DIR)
DRIVE_OUT="/content/drive/MyDrive/planting_outputs"   # where EE table exports land
print("pipeline on path:", PIPE_DIR)

ModuleNotFoundError: No module named 'google.colab'

### Stage 0d · Shared scoring helpers

**What these do.** `find_csv` resolves a glob across both folders and returns the **newest** match.
That newest-first rule is not cosmetic: Earth Engine never overwrites, so a re-export lands as
`name (1).csv` while the stale file keeps the clean name. Taking the newest is what stops a re-run
silently scoring the previous run's numbers.

The rest are the scoring conventions used throughout: leave-one-out cross-validation with every free
parameter refit on $n-1$, a paired bootstrap interval, and Spearman reported beside MAE because rank
skill is invariant to the yield ceiling. With 6 to 81 zones, a single 70/30 split has a spread larger
than any effect measured here.

In [ ]:
# shared scoring helpers used by several tests
import csv, glob, re, json
import numpy as np
from scipy import stats
CY = os.path.join(PIPE_DIR, "Cropyield-Data")

def find_csv(pat):
    hits = [f for d in (CY, DRIVE_OUT) for f in glob.glob(os.path.join(d, pat))]
    return max(hits, key=os.path.getmtime) if hits else None

def norm(s): return re.sub(r"[^a-z0-9]", "", str(s).lower())

def fit_ym_7030(obs, rel, seed=42):
    """Least-squares Ym through the origin on a fixed 70/30 split; held-out MAE/bias/r."""
    rng = np.random.default_rng(seed); idx = rng.permutation(len(obs))
    ntr = int(round(0.7 * len(obs))); tr, te = idx[:ntr], idx[ntr:]
    ym = float(np.sum(rel[tr] * obs[tr]) / np.sum(rel[tr] ** 2)); pred = ym * rel
    return dict(ym=ym, mae=float(np.abs(pred[te]-obs[te]).mean()),
                bias=float((pred[te]-obs[te]).mean()),
                r=float(np.corrcoef(pred[te], obs[te])[0, 1]), ntr=len(tr), nte=len(te))

def loo_mae(obs, rel):
    """Leave-one-out MAE with the Ym refit for every held-out zone."""
    n = len(obs); err = np.empty(n)
    for i in range(n):
        m = np.ones(n, bool); m[i] = False
        ym = np.sum(rel[m]*obs[m]) / np.sum(rel[m]**2)
        err[i] = abs(ym*rel[i] - obs[i])
    return err

def paired_bootstrap_ci(dA, dB, nboot=2000, seed=7):
    """95% CI on mean(dB − dA) over the same zones."""
    d = dB - dA; rng = np.random.default_rng(seed)
    boots = np.array([d[rng.integers(0, len(d), len(d))].mean() for _ in range(nboot)])
    return float(d.mean()), tuple(np.percentile(boots, [2.5, 97.5]))

def skill_line(name, obs, pred):
    e = pred - obs
    rho = stats.spearmanr(pred, obs).statistic
    print(f"  {name:<14} MAE {np.abs(e).mean():.3f}  bias {e.mean():+.3f}  "
          f"r {np.corrcoef(pred,obs)[0,1]:+.2f}  rho {rho:+.2f}")
print("helpers ready")

### Stage 0e · Poll the export queue

Prints the state of the most recent tasks. Re-run to refresh. More than about 20 minutes with nothing
entering RUNNING is a stalled queue, not a slow one; cancel, switch project, resubmit.

In [ ]:
# optional: poll EE export tasks from the notebook
def ee_tasks(n=8):
    for t in ee.batch.Task.list()[:n]:
        print(t.status().get("state","?"), "·", t.config.get("description","?"))
ee_tasks()

## T1 · Fusion for onset: cue fusion vs ubESTARFM (RETIRED)
**Question.** Does ubESTARFM (Sentinel-2 blended with dense MODIS reflectance) beat the production cue fusion for onset at a matched 250 m?
**Verdict on record.** No. Calendar hit-rate 81.5% (cue) vs 77.3% raw fill and 78.4% calibrated. MODIS has no red edge, so the fill is NDVI, which rises early and pulls SOS about one dekad early. ubESTARFM is retired; code kept for reproducibility. Full finding: `Documentation_ALL_2026-08-09/UBESTARFM_FINDING.md`.

### T1 · Cue fusion against ubESTARFM — retired

**On record.** Cue fusion **81.5 %** against ubESTARFM **77.3 to 78.4 %** at 250 m. The blended
reflectance product did not beat the simpler multi-cue fusion for onset detection, at a much higher
compute cost.

**Expected output.** The skipped message and that finding. The module is kept in `src/estarfm.py` if
the result ever needs reproducing; set the flag to `True` only then, because the run is heavy.

**Why a negative result is kept.** It is the reason the production onset is cue fusion. Without it, the
same idea would be proposed again.

In [ ]:
# re-run only if you need to reproduce the retired result (heavy). The module is kept in src/estarfm.py.
RERUN_UBESTARFM = False
if RERUN_UBESTARFM:
    from src import estarfm as EF
    help(EF)
else:
    print("skipped. Finding on record: cue 81.5% vs ubESTARFM 77.3-78.4% at 250 m. RETIRED.")

## T2 · Resolution ablation: 10 m vs 250 m
**Question.** How much onset skill does the operational 250 m scale cost against a native 10 m run of the same method?
**Verdict on record.** 92.7% hit-rate at 10 m vs 81.5% at 250 m: −11.2 points. Resolution dominates every fusion choice. Two concurrent 10 m country exports do not sustain on GEE, so 250 m is the production scale.

### T2 · 10 m against 250 m

**The question.** How much onset skill does the operational 250 m scale cost against a native 10 m run
of the same method?

**To reproduce.** Run `01_planting_window.ipynb` twice over the same box with the scale changed, then
score both with `admin_skill_local.py`. The comparison is only meaningful over a small box; a 10 m run
of a whole country is not feasible interactively.

In [ ]:
# reproduce by running the planting window at both scales over a small box, then scoring vs the calendar.
RERUN_RESOLUTION = False
if RERUN_RESOLUTION:
    print("Run 01_planting_window.ipynb twice with scale=10 and scale=250 over aoi_run,")
    print("then score both with admin_skill_local.py against config/season_calendar.csv.")
else:
    print("skipped. On record: 92.7% (10 m) vs 81.5% (250 m), same method and AOI.")

## T3 · LTN prior ablation
**Question.** Does gating the SOS search with the long-term-normal prior improve planting skill?
**Method.** Run SOS with and without `ltn_sos`, score both against the calendar window at admin-1.
**Verdict on record.** Long rains 91.4% → 95.9% (+4.5 pts), 100% of pixels kept. Short rains 59.8% → 53.4% with 78% kept: the prior keeps the season running but cannot sharpen it. Confirmation-limited, a data limit.

### T3 · Does the climatological prior help?

**The question.** Does gating the start-of-season search to within two dekads of the long-term normal
improve planting skill, or does it just suppress variability?

**Why the gate exists.** Without it, a weed flush or a second green-up can be picked up as the season.
With it, a genuinely anomalous season is clipped toward the normal. The pipeline applies the gate **only
where a prior exists**, passing the calendar window through where it is masked, so a sparse
second-season normal cannot reject every pixel.

**Expected runtime.** Minutes over a test box, when the flag is set to `True`.

In [ ]:
RERUN_LTN_ABLATION = False   # ~minutes over a test box; set True to reproduce
if RERUN_LTN_ABLATION:
    from src import (utils, s2_preprocess as S2, s1_preprocess as S1,
                     fusion_phenometrics as FZ, ltn as LTN, planting_date as PD)
    from run import crop_mask_image, GAUL_NAME
    from src import zonal_aggregate as ZA
    COUNTRY, SEASON, YEAR = "Kenya", "Long rains", 2024
    aoi_run = ee.Geometry.Rectangle([34.4, -1.2, 37.8, 1.2])
    kc, soil = utils.load_crop_coeffs()
    rows = {(r["country"], r["season"]): r for r in utils.viable_products(utils.load_calendar("config/season_calendar.csv")) if r["crop"].lower() == "maize"}
    ss, se = utils.sos_window_dekads(rows[(COUNTRY, SEASON)]["sos_detection_window"])
    mask = crop_mask_image(ee, COUNTRY, "maize", None)
    s2 = S2.build_s2_dekadal(ee, aoi_run, YEAR); s1 = S1.build_s1_dekadal(ee, aoi_run, YEAR, orbit="ASCENDING")
    fpar = FZ.add_fpar_dekadal(ee, aoi_run, YEAR)
    g = FZ.build_fused_greenness(ee, s2, s1, fpar)
    ltn = LTN.build_ltn_prior(ee, aoi_run, ss, se)
    sos_prior = FZ.detect_sos(ee, g, mask, ss, se, ltn_sos=ltn, ltn_pad=2)
    sos_bare  = FZ.detect_sos(ee, g, mask, ss, se)          # ablated: no prior
    for name, sos in [("with prior", sos_prior), ("no prior", sos_bare)]:
        pl = PD.sos_to_planting(ee, sos, "maize")
        h = pl.gte(6).And(pl.lte(12)).reduceRegion(ee.Reducer.mean(), aoi_run, 250, maxPixels=1e12)
        print(name, "in-window share:", h.getInfo())
else:
    print("skipped. On record: +4.5 pts long rains; short rains kept running (78%) but not sharpened.")

## T4 · Growing period: fixed 120 d vs zone-aware LGP
**Question.** Does a zone-aware season length (highland 180 d / bimodal 120 d / warm lowland 90 d) beat the single 120 d on yield skill?
**Method.** County relyield from the CHIRPS MAM water balance at each duration, Ym refit per arm on the same 70/30 split, scored on the held-out 30% vs HarvestStat.
**Verdict.** Fixed 120 d wins (MAE 0.568, r 0.66 vs 0.660, r 0.51). A 180 d crop on the MAM window accumulates deficit after May and degrades the highland ranking. The zone window idea returns properly in T8.

### T4 · Fixed 120 days against a zone-aware season length

**On record: keep the fixed 120 days.** The zone-aware arm loses on MAE, 0.684 against 0.601, and
collapses the ranking, Spearman 0.525 against 0.725. Across 500 seeds it wins in only 10 % of them.

**Expected output.** The scores if the per-duration CSVs are present, otherwise a message saying the
recorded verdict stands, with the write-up in `Cropyield-Data/lgp_ab_test_MAM.md`.

**Read this with T8.** T4 says a zone-aware **length** does not improve yield. T8 says the fixed length
is wrong by 68 days in the highlands. Both are true: the fixed cycle is wrong, and the particular
zone-aware replacement tested was not an improvement.

In [ ]:
# inputs: maize_wkt_kenya_wb_MAM_{90,120,180}d_masked.csv (per-duration water-balance relyield)
p120 = find_csv("maize_wkt_kenya_wb_MAM_120d_masked*.csv")
if p120 is None:
    print("duration CSVs not found; the recorded verdict stands (lgp_ab_test_MAM.md in Cropyield-Data)")
else:
    import pandas as pd
    hs = {norm(r["admin_1"]): float(r["obs_mean"]) for r in csv.DictReader(open(os.path.join(CY, "harveststat_obs_KE_Long.csv"))) if r.get("obs_mean")}
    def county_rel(path):
        df = pd.read_csv(path)
        g = df.groupby("county")["relyield"].mean()
        return {norm(k): v for k, v in g.items()}
    rel120 = county_rel(p120)
    # zone-aware: highland >=1800 m uses the 180 d file, <1000 m the 90 d file, else 120 d
    rel90  = county_rel(find_csv("maize_wkt_kenya_wb_MAM_90d_masked*.csv"))
    rel180 = county_rel(find_csv("maize_wkt_kenya_wb_MAM_180d_masked*.csv"))
    elev = json.load(open(os.path.join(PIPE_DIR, "config", "county_elev.json"))) if os.path.exists(os.path.join(PIPE_DIR, "config", "county_elev.json")) else {}
    keys = sorted(set(rel120) & set(hs))
    obs = np.array([hs[k] for k in keys])
    a = np.array([rel120[k] for k in keys])
    def zone_rel(k):
        e = elev.get(k, 1400)
        return (rel180 if e >= 1800 else rel90 if e < 1000 else rel120).get(k, rel120[k])
    b = np.array([zone_rel(k) for k in keys])
    print("A fixed 120 d :", fit_ym_7030(obs, a))
    print("B zone LGP    :", fit_ym_7030(obs, b))

## T5 · Ym recalibration against HarvestStat (70/30)
**Question.** What potential-yield ceiling fits observed admin yields?
**Method.** `harveststat_yield_validation.py` builds the obs tables; here the fit is reproduced inline per product.
**Verdict.** KE Long rains 6.0 → 2.56 (held-out MAE 2.52 → 0.58, r +0.64). KE Short rains 4.5 → 2.1. ET Meher 6.0 → 4.6 (9 regions, provisional).

### T5 · Fitting the yield ceiling to HarvestStat

**What this does.** Fits $Y_m$ so that $Y_a = (\mathrm{CPI}/100)\,Y_m$ matches reported sub-national
yields, as a least-squares slope through the origin, tested out of sample.

**The current fit is the typical-year one**, with the median over the available years as the target:
Kenya 2.34 long rains and 1.44 short rains, Ethiopia 4.14, Rwanda 2.61, Burundi 1.88, Somalia 1.02,
Uganda 2.34 provisional. Held-out error falls by 49 to 87 % against the uncalibrated defaults.

**The distinction that matters for use.** Where $r$ is near zero, as in Rwanda, Burundi and Somalia, the
ceiling fixes the **level** only. Those maps are usable for national and seasonal totals and not for
ranking districts. Full write-up in `yield_calibration_2024/YIELD_CALIBRATION_2024.md`.

In [ ]:
# rebuild obs tables from HarvestStat if needed (writes Cropyield-Data/harveststat_obs_*.csv)
REBUILD_OBS = False
if REBUILD_OBS:
    exec(open("harveststat_yield_validation.py").read())
for tag, obsfn, relsrc, keycol, relcol in [
        ("KE_Long",  "harveststat_obs_KE_Long.csv",  "whc_ab_county_scores.csv",       "county", "cpi_B_rel"),
        ("KE_Short", "harveststat_obs_KE_Short.csv", "whc_ab_short_county_scores.csv", "zone",   "cpi_B_rel"),
        ("ET_Meher", "harveststat_obs_ET_Meher.csv", "whc_ab_et_meher_scores.csv",     "zone",   "cpi_B_rel")]:
    relp = os.path.join(CY, relsrc)
    if not os.path.exists(relp):
        print(tag, "rel table missing (run T7 GEE stage first)"); continue
    R = list(csv.DictReader(open(relp)))
    okey = "obs_t_ha" if "obs_t_ha" in R[0] else "obs_mean_t_ha"
    obs = np.array([float(r[okey]) for r in R]); rel = np.array([float(r[relcol]) for r in R])
    print(tag, fit_ym_7030(obs, rel))

## T6 · Per-zone Ym: single national vs highland split
**Question.** Is the highland yield edge potential (a higher ceiling) rather than season length?
**Method.** Same 70/30 protocol; arm B gives counties at ≥ 1800 m Ym 3.7 and the rest 2.3.
**Verdict.** B wins, held-out MAE 0.568 → 0.472. Applied: `src/cpi.py ym_img_for`, wired in `run_cpi.py` (run `cpi_Kenya_Longrains_2024_aezym`: bias +0.04, MAE 0.61).

### T6 · One national ceiling against a highland split

**The question.** Is the highland yield advantage potential, meaning a higher ceiling, rather than a
longer season?

**On record: the highland lever is the ceiling, but the evidence supports the ranking claim only.** The
per-zone ceiling improved leave-one-out MAE by 0.063 with an interval of [−0.046, +0.167], which is not
significant once the second free parameter is paid for; the out-of-sample **ranking** gain is real.

**Expected output.** The current scalar ceilings, then the per-zone image. `YM_HIGHLAND` is **empty** in
`src/cpi.py`, so `ym_img_for` returns a constant image: the split is off under the typical-year
calibration, which uses one ceiling per country and season. The 2024 values are kept in a comment.

In [ ]:
from src.cpi import ym_for, ym_img_for
print("current scalar ceilings:", [(c, s, ym_for(c, s)) for c, s in
      [("Kenya","Long rains"), ("Kenya","Short rains"), ("Ethiopia","Meher")]])
# the per-zone image (highland >= 1800 m -> 3.7, else 2.3):
from run import GAUL_NAME
from src import zonal_aggregate as ZA
aoi = ZA.gaul_admin(ee, [GAUL_NAME["Kenya"]], level=0).geometry()
ym_img = ym_img_for(ee, aoi, "Kenya", "Long rains")
print("per-zone Ym image:", ym_img.bandNames().getInfo())
print("Uasin Gishu / Garissa spot check should read 3.7 / 2.3 (see run_cpi.py)")

## T7 · Soil bucket: uniform 100 mm vs SoilGrids/Saxton WHC
**Question.** Does the per-pixel root-zone bucket beat a flat 100 mm?
**Method.** Identical pipeline, only `whc_img` differs. GEE stage exports county CPI per arm; scoring uses leave-one-out MAE with a per-arm Ym refit, plus a paired bootstrap CI.
**Verdict.** Long rains: a tie (Δ −0.011, CI spans 0). Short rains county: a significant win (Δ −0.046, CI −0.086 to −0.007). Keep the SoilGrids bucket everywhere; the short rains is its evidence. Protocol: `WHC_AB_PROTOCOL.md`.

### T7 · Uniform 100 mm against SoilGrids water-holding capacity

**On record: null, and worse in the case that matters.** Under a **shared** ceiling, SoilGrids never
wins: Kenya long −0.024, Kenya short +0.010, wards 2021 −0.082, and Kitui 2022 **−0.318**. The one
positive result vanished once both arms shared a ceiling, because SoilGrids is a uniformly bigger
bucket and its dominant effect is a level shift that a per-arm ceiling absorbs by construction.

**The Kitui case is the substantive one.** The bigger bucket carries enough stored water through the dry
spell that the model does not register the 2022 drought. SoilGrids stays the default on physical
grounds, but it cannot be claimed to improve skill.

**Expected output.** The four score tables. Exports are off by default; scoring reads the CSVs.

In [ ]:
# GEE stage (starts exports; wait for them in the Tasks panel, then run the scoring cell)
RUN_WHC_EXPORTS = False
if RUN_WHC_EXPORTS:
    exec(open("whc_ab_test.py").read())        # Kenya Long rains, county
    # variants (short-rains county + wards, Ethiopia): see whc_ab_variants.py
    # exec(open("whc_ab_variants.py").read())
else:
    print("skipped exports; scoring below uses the CSVs already in Cropyield-Data/ or Drive")

In [ ]:
for label, fn in [("KE Long rains county", "whc_ab_county_scores.csv"),
                  ("KE Short rains county", "whc_ab_short_county_scores.csv"),
                  ("KE SR wards pooled", "whc_ab_ward_scores_pooled.csv"),
                  ("ET Meher regions", "whc_ab_et_meher_scores.csv")]:
    p = os.path.join(CY, fn)
    if not os.path.exists(p):
        print(label, ": scores file missing"); continue
    R = list(csv.DictReader(open(p)))
    okey = "obs_t_ha" if "obs_t_ha" in R[0] else "obs_mean_t_ha"
    obs = np.array([float(r[okey]) for r in R])
    a = np.array([float(r["cpi_A_rel"]) for r in R]); b = np.array([float(r["cpi_B_rel"]) for r in R])
    eA, eB = loo_mae(obs, a), loo_mae(obs, b)
    dm, (lo, hi) = paired_bootstrap_ci(eA, eB)
    sig = " ***" if (hi < 0 or lo > 0) else ""
    print(f"{label:<24} n={len(R):>3}  LOO-MAE A {eA.mean():.3f}  B {eB.mean():.3f}  "
          f"dB-A {dm:+.3f}  CI [{lo:+.3f}, {hi:+.3f}]{sig}")

## T8 · Season length: fixed 120 d vs the GDD thermal clock
**Question.** How far is the fixed 120 d from each pixel's thermal season?
**Method.** `lgp_vs_gdd.py` exports per-zone GDD season length and flowering timing; the scorer rolls districts up to counties.
**Verdict.** Kenya long rains: county mean 137 d, pixel-weighted 173 d; flowering lands at ~87 d, not 70. Only 8/44 counties sit within ±15 d of 120. The highlands need their own unimodal window (March to August). Meher runs ~154 d pixel-weighted.

### T8 · Fixed 120 days against the GDD thermal clock

**On record, and the largest open discrepancy in the pipeline.** Pixel-weighted over actual maize the
thermal season is **173 days** against the fixed 120, and only **8 of 44** Kenyan counties fall within
±15 days. Highlands run **+67.7 days** long with flowering **25.9 days later**; hot arid lands run
short. Ethiopia Meher is +22 days.

Because the stage weights put three times the weight on flowering, the water balance is being read at
the wrong point of the season over most of Kenya's maize. The recommendation on record is to replace the
fixed cycle, and **not** to adopt the current maturity targets uncritically: the CHIRTS work found the
spreadsheet targets worse than the ERA5 ones.

In [ ]:
RUN_LGP_EXPORTS = False
if RUN_LGP_EXPORTS:
    for v in ["ke_long", "ke_short", "et_meher"]:
        os.system(f"python lgp_vs_gdd.py --variant {v}")
!python lgp_vs_gdd_score.py || echo "(needs the lgp_vs_gdd_* exports in Drive/planting_outputs)"

## T9 · Biomass route: DMP-derived yield vs the CPI yield
**Question.** Can a dry-matter-productivity route (MODIS GPP stand-in for Copernicus DMP) estimate yield?
**Method.** `dmp_run.py` integrates DM over each pixel's crop cycle and exports zonal tables; `dmp_score.py` scores DMP and CPI against HarvestStat and the crop cuts, and back-solves the harvest index the observations would need.
**Verdict.** Ranking is good (ρ +0.77 long rains, +0.64 short rains, beats CPI in both) but the level runs 1.6 to 1.7 times hot; the crop-cut wards show 8.2 times over with implied HI 0.06, so 500 m GPP cannot see plot-scale failure. Use as a ranking covariate, not a level product.

### T9 · The biomass route against the water-balance route

**On record: DMP out-ranks CPI in all five variants**, and the gap is widest where CPI ranks
**backwards**, Kitui 2022 at ρ −0.32 and Ethiopia Meher at −0.40.

**But the level is unusable as it stands.** DMP over-predicts by 1.6 to 23.5 times and implies a harvest
index of 0.019 in the Kitui wards, because 500 m pixels carry non-crop biomass. Notebook 05 therefore
tests DMP as an **anomaly** rather than as raw biomass.

**Ethiopia Meher is the actionable case**: DMP wins on every metric and implies a harvest index of 0.468,
inside the agronomic range.

**The source is a stand-in.** Copernicus DMP is not in the Earth Engine catalog, so MODIS MOD17A2H GPP
is used, converted through a carbon-use efficiency of 0.45 and a carbon fraction of 0.475. Report the
rankings, not the magnitudes.

In [ ]:
RUN_DMP_EXPORTS = False
if RUN_DMP_EXPORTS:
    exec(open("dmp_run.py").read())            # starts the five zonal exports
!python dmp_score.py || echo "(needs the dmp_yield_* exports in Drive/planting_outputs)"

## T10 · Planting dekad vs farmer records
**Question.** How close are the model's planting dekads to what ~4.8 M farmers reported?
**Verdict.** Kenya MAM county: modal bias −0.31 dk, MAE 1.02 dk, 92.9% within ±2 (42/44 counties). Ward: 855 wards, MAE 1.10, 96% within ±2. Short rains not yet cleanly validated: the survey predates OND and the west plants an August to September second season three dekads before the OND window. Ethiopia has no farmer records yet. Full write-up: `PLANTING_VALIDATION_2024.md`.

### T10 · Planting dekad against farmer records

**The strongest validation in the whole set**, because it compares against what farmers reported rather
than against another model or a calendar.

**Expected values.** **42 counties**, modal bias **−0.31 dekads**, MAE **1.02 dekads**, **93 %** within
two dekads. At ward level, **855 wards** across the same counties, bias −0.38, MAE 1.09, **96 %** within
two dekads.

**What it licenses.** Ward-level error is essentially the same as county-level error, which means the
residual is the genuine spread of planting within a season, not an aggregation artefact. Treat one dekad
as the noise floor: a one-dekad difference between two units is not a finding, a three-dekad difference
is.

**And what it does not.** This is Kenya long rains 2024 only. No other country or season has farmer
records behind it, and the planting estimate elsewhere carries only the calendar comparison, which is a
much weaker test.

In [ ]:
R = list(csv.DictReader(open(os.path.join(PIPE_DIR, "planting_validation_MAM_2024.csv"))))
err = np.array([float(r["err_modal"]) for r in R])
print(f"counties {len(R)}  modal bias {err.mean():+.2f} dk  MAE {np.abs(err).mean():.2f} dk  "
      f"within ±1 {100*np.mean(np.abs(err)<=1):.1f}%  within ±2 {100*np.mean(np.abs(err)<=2):.1f}%")
worst = sorted(R, key=lambda r: -abs(float(r["err_modal"])))[:5]
print("largest misses:", ", ".join(f"{r['County'].title()} {float(r['err_modal']):+.0f}" for r in worst))

## Provenance
Scripts these sections call or reproduce: `whc_ab_test.py`, `whc_ab_variants.py`, `whc_ab_score.py`, `whc_ab_score_variants.py`, `lgp_vs_gdd.py`, `lgp_vs_gdd_score.py`, `dmp_run.py`, `dmp_score.py`, `src/dmp_yield.py`, `harveststat_yield_validation.py`, `src/cpi.py`, `src/soil.py`, `src/gdd_clock.py`, `src/estarfm.py` (retired). Results narrative with figures: `Pipeline_Workflow_Methodologies.docx` section 10.